# H-MIL Final Training — 3-Seed Ensemble

Fixed hyperparameters from v3 Trial 0 (best QWK=0.5861, F1=0.6296).

Trains 3 independent models with seeds 42, 7, 123.
Each runs for full 60 epochs (no Optuna pruner, extended from 40).
All 3 weight files saved. Ensemble QWK/F1 reported at end.

Outputs (all downloaded automatically):
- `seed42_weights.msgpack`
- `seed7_weights.msgpack`  
- `seed123_weights.msgpack`
- `final_ensemble_results.json`  (per-seed + ensemble metrics)
- `val_predictions.json`  (raw predictions for heatmap script)

In [7]:
# Do NOT reinstall jax/jaxlib — overwrites TPU build
!pip install -q -U flax optax scikit-learn

import jax
print(f"Backend: {jax.default_backend()}")
print(f"Devices: {jax.device_count()}")
assert jax.device_count() == 8, "TPU not detected."

/usr/local/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Backend: tpu
Devices: 8


In [8]:
%%writefile dataset_utils.py
import csv
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset

MAX_SAFE_CEILING = 10000
NUM_WORKERS      = 4


class SlideDatasetPT(Dataset):
    def __init__(self, labels_csv, pt_dir, split_ids):
        self.pt_dir = Path(pt_dir)
        with open(labels_csv) as f:
            all_rows = list(csv.DictReader(f))
        split_set = set(split_ids)
        self.samples = []
        for row in all_rows:
            pid = row["patient_id"]
            if pid not in split_set:
                continue
            raw_label = int(row["label"])
            assert raw_label in {0, 1, 2}, f"Corrupt label {raw_label} for {pid}"
            pt_path = self.pt_dir / f"{pid}.pt"
            if pt_path.exists():
                self.samples.append({
                    "patient_id": pid,
                    "label":      raw_label,
                    "pt_path":    pt_path,
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s    = self.samples[idx]
        data = torch.load(s["pt_path"], weights_only=True)
        return data["features"], s["label"], data["coords"], s["patient_id"]


def train_collate_fn(batch):
    """Stochastic pseudo-bag — fresh randperm every call."""
    features, labels, _, __ = zip(*batch)
    B, D = len(features), features[0].shape[1]
    padded = np.zeros((B, MAX_SAFE_CEILING, D), dtype=np.float32)
    masks  = np.zeros((B, MAX_SAFE_CEILING),    dtype=bool)
    for i, f in enumerate(features):
        n = f.shape[0]
        if n > MAX_SAFE_CEILING:
            idx       = torch.randperm(n)[:MAX_SAFE_CEILING].numpy()
            padded[i] = f.numpy()[idx]
            masks[i]  = True
        else:
            padded[i, :n] = f.numpy()
            masks[i,  :n] = True
    return padded, np.array(labels, dtype=np.int32), masks


def val_collate_fn(batch):
    """Deterministic — always takes first MAX_SAFE_CEILING patches.
    Returns coords and patient IDs for heatmap script."""
    features, labels, coords, pids = zip(*batch)
    B, D = len(features), features[0].shape[1]
    padded     = np.zeros((B, MAX_SAFE_CEILING, D), dtype=np.float32)
    masks      = np.zeros((B, MAX_SAFE_CEILING),    dtype=bool)
    coords_out = []
    for i, (f, c) in enumerate(zip(features, coords)):
        n = f.shape[0]
        if n > MAX_SAFE_CEILING:
            padded[i]  = f.numpy()[:MAX_SAFE_CEILING]
            masks[i]   = True
            coords_out.append(c.numpy()[:MAX_SAFE_CEILING])
        else:
            padded[i, :n] = f.numpy()
            masks[i,  :n] = True
            coords_out.append(c.numpy())
    return padded, np.array(labels, dtype=np.int32), masks, coords_out, list(pids)

Overwriting dataset_utils.py


In [9]:
"""
Final 3-seed training.
No Optuna. Fixed config from v3 Trial 0.
Trains 60 epochs per seed with patience=15 (relaxed: we want full convergence).
"""

import os
import csv
import json
import shutil
import collections
import multiprocessing as mp
from pathlib import Path
import numpy as np

import jax
import jax.numpy as jnp
from jax import random
import flax
import flax.linen as nn
from flax.training import train_state
from flax.training.common_utils import shard
from flax import serialization
import optax
from sklearn.metrics import f1_score, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import DataLoader
from dataset_utils import (
    SlideDatasetPT, train_collate_fn, val_collate_fn,
    MAX_SAFE_CEILING, NUM_WORKERS
)

print(f"Backend: {jax.default_backend()}  |  Devices: {jax.device_count()}")


# ==========================================================================
# FIXED CONFIG — from v3 Trial 0
# ==========================================================================
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/apexblue/tcga-hnsc-pt-embeddings"
LABELS_CSV         = "/kaggle/input/datasets/apexblue/tumour-grading-model-tcga-manifest/tcga_hnsc_labels.csv"
OUTPUT_DIR         = "/kaggle/working/final_models"

# Searched hyperparameters — v3 Trial 0 best
BASE_LR         = 0.00028078987855609705
LR_DECAY_GLOBAL = 0.6235928598332892
LR_DECAY_LOCAL  = 0.7342380613796495
DROPOUT_RATE    = 0.18276391449291166
ALPHA           = 0.50   # ordinal loss coefficient

# Fixed config
EMBED_DIM    = 1024
N_CLASSES    = 3
BATCH_SIZE   = 8
HIDDEN_DIM   = 256
WEIGHT_DECAY = 4.550475813202181e-05
EPOCHS       = 60   # Extended: no pruner pressure, let it fully converge
PATIENCE     = 10   # Relaxed: we know the config works, give it room
GRADE_VALUES = jnp.array([0.0, 1.0, 2.0])
SEEDS        = [7, 123]

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


# ==========================================================================
# DATA SPLIT
# ==========================================================================
def get_splits(labels_csv, seed=42):
    with open(labels_csv) as f:
        rows = list(csv.DictReader(f))
    seen = {}
    for r in rows:
        if r["patient_id"] not in seen:
            seen[r["patient_id"]] = int(r["label"])
    pids, labels = list(seen.keys()), list(seen.values())
    return train_test_split(pids, test_size=0.20, stratify=labels, random_state=42)

TRAIN_IDS, VAL_IDS = get_splits(LABELS_CSV)
print(f"Split: {len(TRAIN_IDS)} train / {len(VAL_IDS)} val")


# ==========================================================================
# ARCHITECTURE
# ==========================================================================
class HMIL_Flax(nn.Module):
    hidden_dim:   int
    n_classes:    int   = 3
    region_size:  int   = 100
    dropout_rate: float = 0.3

    @nn.compact
    def __call__(self, x, mask, deterministic: bool):
        B, N, D = x.shape
        h = nn.Dense(self.hidden_dim, name='proj')(x)
        h = nn.relu(h)

        pad_len = (self.region_size - (N % self.region_size)) % self.region_size
        if pad_len > 0:
            h    = jnp.pad(h,    ((0, 0), (0, pad_len), (0, 0)))
            mask = jnp.pad(mask, ((0, 0), (0, pad_len)))
        num_regions = h.shape[1] // self.region_size

        h_reg = h.reshape((B * num_regions, self.region_size, self.hidden_dim))
        m_reg = mask.reshape((B * num_regions, self.region_size))
        lcls  = self.param('local_cls', nn.initializers.normal(0.02),
                           (1, 1, self.hidden_dim))
        h_reg = jnp.concatenate(
            [jnp.broadcast_to(lcls, (B * num_regions, 1, self.hidden_dim)), h_reg], axis=1
        )
        lmask = jnp.concatenate(
            [jnp.ones((B * num_regions, 1), dtype=jnp.bool_), m_reg], axis=1
        )
        h_reg = h_reg + nn.SelfAttention(
            num_heads=4, qkv_features=self.hidden_dim,
            dropout_rate=self.dropout_rate, name='local_trans'
        )(nn.LayerNorm(name='local_ln')(h_reg),
          mask=jnp.expand_dims(lmask, (1, 2)), deterministic=deterministic)

        h_pool = h_reg[:, 0, :].reshape((B, num_regions, self.hidden_dim))

        gvalid = m_reg.any(axis=1).reshape((B, num_regions))
        gcls   = self.param('global_cls', nn.initializers.normal(0.02),
                            (1, 1, self.hidden_dim))
        h_glob = jnp.concatenate(
            [jnp.broadcast_to(gcls, (B, 1, self.hidden_dim)), h_pool], axis=1
        )
        gmask = jnp.concatenate(
            [jnp.ones((B, 1), dtype=jnp.bool_), gvalid], axis=1
        )
        h_glob = h_glob + nn.SelfAttention(
            num_heads=4, qkv_features=self.hidden_dim,
            dropout_rate=self.dropout_rate, name='global_trans'
        )(nn.LayerNorm(name='global_ln')(h_glob),
          mask=jnp.expand_dims(gmask, (1, 2)), deterministic=deterministic)

        z = nn.Dense(self.hidden_dim // 2, name='classifier_1')(h_glob[:, 0, :])
        z = nn.relu(z)
        z = nn.Dropout(self.dropout_rate)(z, deterministic=deterministic)
        return nn.Dense(self.n_classes, name='classifier_2')(z)


# ==========================================================================
# HELPERS
# ==========================================================================
def replicate(x):
    return jax.tree_util.tree_map(lambda a: jnp.stack([a] * jax.device_count()), x)


def build_tx(params, n_steps_per_epoch):
    warmup   = 5  * n_steps_per_epoch
    total_st = EPOCHS * n_steps_per_epoch
    lr_g     = BASE_LR * LR_DECAY_GLOBAL
    lr_l     = lr_g    * LR_DECAY_LOCAL
    lr_p     = lr_l    * 0.5

    def make_sched(peak):
        return optax.warmup_cosine_decay_schedule(
            init_value=0.0, peak_value=peak,
            warmup_steps=warmup, decay_steps=total_st,
            end_value=peak * 0.01
        )

    flat   = flax.traverse_util.flatten_dict(params)
    labels = flax.traverse_util.unflatten_dict({
        k: ('classifier'   if 'classifier' in '/'.join(k) else
            'global_layer' if 'global'     in '/'.join(k) else
            'local_layer'  if 'local'      in '/'.join(k) else
            'proj_layer')
        for k in flat
    })
    return optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.multi_transform(
            {
                'classifier':   optax.adamw(make_sched(BASE_LR), weight_decay=WEIGHT_DECAY),
                'global_layer': optax.adamw(make_sched(lr_g),    weight_decay=WEIGHT_DECAY),
                'local_layer':  optax.adamw(make_sched(lr_l),    weight_decay=WEIGHT_DECAY),
                'proj_layer':   optax.adamw(make_sched(lr_p),    weight_decay=WEIGHT_DECAY),
            },
            param_labels=labels
        )
    )


def run_eval(eval_step_pmap, state_params, val_loader):
    """Returns predictions, labels, softmax probs, pids, coords."""
    all_preds, all_labels, all_probs, all_pids, all_coords = [], [], [], [], []
    for batch in val_loader:
        feats, labels, masks, coords, pids = batch
        n   = labels.shape[0]
        rem = n % jax.device_count()
        if rem != 0:
            pad   = jax.device_count() - rem
            feats = np.pad(feats, ((0, pad), (0, 0), (0, 0)))
            masks = np.pad(masks, ((0, pad), (0, 0)))
        preds, probs = eval_step_pmap(state_params, shard(feats), shard(masks))
        all_preds.extend(np.array(preds).flatten()[:n])
        all_probs.extend(np.array(probs).reshape(-1, N_CLASSES)[:n])
        all_labels.extend(labels)
        all_pids.extend(pids)
        all_coords.extend(coords)
    return all_preds, all_labels, all_probs, all_pids, all_coords


# ==========================================================================
# TRAINING FUNCTION
# ==========================================================================
def train_one_seed(seed):
    print(f"\n{'#'*65}")
    print(f"SEED {seed}")
    print(f"{'#'*65}")

    train_ds = SlideDatasetPT(LABELS_CSV, KAGGLE_DATASET_DIR, TRAIN_IDS)
    val_ds   = SlideDatasetPT(LABELS_CSV, KAGGLE_DATASET_DIR, VAL_IDS)

    counts = collections.Counter(s["label"] for s in train_ds.samples)
    total  = len(train_ds.samples)
    cw     = jnp.array(
        [total / (N_CLASSES * counts.get(c, 1)) for c in range(N_CLASSES)],
        dtype=jnp.float32
    )
    print(f"Class weights → G1:{cw[0]:.2f}  G2:{cw[1]:.2f}  G3:{cw[2]:.2f}")

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=train_collate_fn, num_workers=NUM_WORKERS,
        persistent_workers=True, prefetch_factor=2, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=val_collate_fn, num_workers=NUM_WORKERS,
        persistent_workers=True, prefetch_factor=2, drop_last=False
    )

    model  = HMIL_Flax(hidden_dim=HIDDEN_DIM, dropout_rate=DROPOUT_RATE)
    rng    = random.PRNGKey(seed)
    rng, init_rng = random.split(rng)

    dummy_x    = jnp.ones((BATCH_SIZE, MAX_SAFE_CEILING, EMBED_DIM))
    dummy_mask = jnp.ones((BATCH_SIZE, MAX_SAFE_CEILING), dtype=jnp.bool_)
    params = model.init(init_rng, dummy_x, dummy_mask, deterministic=True)['params']

    tx     = build_tx(params, len(train_loader))
    state  = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)
    state  = replicate(state)
    cw_rep = replicate(cw)
    gv_rep = replicate(GRADE_VALUES)

    # Train step
    def train_step(state, feats, masks, labels, weights, grade_vals, rng_drop):
        def loss_fn(params):
            logits  = state.apply_fn(
                {'params': params}, feats, masks,
                deterministic=False, rngs={'dropout': rng_drop}
            )
            one_hot = jax.nn.one_hot(labels, N_CLASSES)
            ce      = optax.softmax_cross_entropy(logits=logits, labels=one_hot)
            wce     = jnp.mean(ce * weights[labels])
            probs   = jax.nn.softmax(logits, axis=-1)
            exp_g   = jnp.sum(probs * grade_vals, axis=-1)
            true_g  = labels.astype(jnp.float32)
            penalty = jnp.mean((exp_g - true_g) ** 2)
            return wce + ALPHA * penalty
        grads = jax.grad(loss_fn)(state.params)
        grads = jax.lax.pmean(grads, axis_name='batch')
        return state.apply_gradients(grads=grads)

    train_step_pmap = jax.pmap(train_step, axis_name='batch')

    # Eval step — also returns softmax probs for ensemble
    def eval_step(params, feats, masks):
        logits = model.apply(
            {'params': params}, feats, masks, deterministic=True
        )
        probs = jax.nn.softmax(logits, axis=-1)
        return jnp.argmax(probs, axis=-1), probs

    eval_step_pmap = jax.pmap(eval_step, axis_name='batch')

    best_qwk    = -1.0
    best_f1     = 0.0
    no_improve  = 0
    best_params = None

    for epoch in range(1, EPOCHS + 1):
        for feats, labels, masks in train_loader:
            rng, step_rng = random.split(rng)
            step_rngs = random.split(step_rng, jax.device_count())
            state = train_step_pmap(
                state, shard(feats), shard(masks), shard(labels),
                cw_rep, gv_rep, step_rngs
            )

        preds, labels_val, probs, pids, coords = run_eval(
            eval_step_pmap, state.params, val_loader
        )
        val_f1  = f1_score(labels_val, preds, average='macro')
        val_qwk = cohen_kappa_score(labels_val, preds, weights='quadratic')

        print(
            f"Ep {epoch:02d}/{EPOCHS} "
            f"| F1:{val_f1:.4f} | QWK:{val_qwk:.4f} "
            f"| Best:{max(best_qwk,val_qwk):.4f} "
            f"| Pat:{PATIENCE-no_improve}"
        )

        if val_qwk > best_qwk:
            best_qwk    = val_qwk
            best_f1     = val_f1
            no_improve  = 0
            best_params = jax.tree_util.tree_map(lambda x: x[0], state.params)
            # Save best probs/preds for this seed's best epoch
            best_preds  = preds
            best_probs  = probs
            best_pids   = pids
            best_coords = coords
            best_labels = labels_val
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"  Early stop at epoch {epoch}.")
            break

    print(f"Seed {seed} final → QWK:{best_qwk:.4f}  F1:{best_f1:.4f}")

    # Confusion matrix
    cm = confusion_matrix(best_labels, best_preds)
    print(f"Confusion matrix (rows=true, cols=pred):")
    print(f"  G1: {cm[0]}")
    print(f"  G2: {cm[1]}")
    print(f"  G3: {cm[2]}")

    # Save weights
    weight_path = f"{OUTPUT_DIR}/seed{seed}_weights.msgpack"
    with open(weight_path, "wb") as f:
        f.write(serialization.to_bytes(best_params))
    print(f"  Weights → {weight_path}")

    return {
        "seed":        seed,
        "qwk":         best_qwk,
        "f1":          best_f1,
        "preds":       [int(p) for p in best_preds],
        "probs":       [[float(x) for x in p] for p in best_probs],
        "labels":      [int(l) for l in best_labels],
        "patient_ids": best_pids,
        # coords: list of arrays → convert to lists for JSON
        "coords":      [c.tolist() for c in best_coords],
        "confusion_matrix": cm.tolist(),
    }


# ==========================================================================
# MAIN — run all 3 seeds, compute ensemble
# ==========================================================================
if __name__ == "__main__":
    mp.set_start_method('spawn', force=True)

    print("\n" + "="*65)
    print("H-MIL FINAL TRAINING — 3 SEEDS")
    print(f"Config: lr={BASE_LR:.3e}  dr={DROPOUT_RATE:.3f}  alpha={ALPHA}")
    print(f"        global_decay={LR_DECAY_GLOBAL:.3f}  local_decay={LR_DECAY_LOCAL:.3f}")
    print(f"Epochs: {EPOCHS}  Patience: {PATIENCE}")
    print("="*65)

    seed_results = []
    for seed in SEEDS:
        result = train_one_seed(seed)
        seed_results.append(result)

    # ── Ensemble: average softmax probabilities across seeds ──────────────
    # All seeds evaluated on identical val set (same split, deterministic collate)
    # so patient_ids are aligned. Verify alignment:
    pids_0 = seed_results[0]["patient_ids"]
    for r in seed_results[1:]:
        assert r["patient_ids"] == pids_0, "Patient ID mismatch across seeds!"

    n_val = len(pids_0)
    ensemble_probs = np.zeros((n_val, N_CLASSES), dtype=np.float64)
    for r in seed_results:
        ensemble_probs += np.array(r["probs"])
    ensemble_probs /= len(SEEDS)

    ensemble_preds  = np.argmax(ensemble_probs, axis=1).tolist()
    true_labels     = seed_results[0]["labels"]
    ensemble_f1     = f1_score(true_labels, ensemble_preds, average='macro')
    ensemble_qwk    = cohen_kappa_score(true_labels, ensemble_preds, weights='quadratic')
    ensemble_cm     = confusion_matrix(true_labels, ensemble_preds)

    print("\n" + "="*65)
    print("ENSEMBLE RESULTS")
    print(f"  QWK: {ensemble_qwk:.4f}")
    print(f"  F1:  {ensemble_f1:.4f}")
    print(f"  Confusion matrix (rows=true, cols=pred):")
    print(f"    G1: {ensemble_cm[0].tolist()}")
    print(f"    G2: {ensemble_cm[1].tolist()}")
    print(f"    G3: {ensemble_cm[2].tolist()}")
    print("="*65)

    # Per-seed summary
    print("\nPer-seed summary:")
    for r in seed_results:
        print(f"  Seed {r['seed']:3d}: QWK={r['qwk']:.4f}  F1={r['f1']:.4f}")

    # ── Save everything ───────────────────────────────────────────────────
    final_record = {
        "config": {
            "base_lr":         BASE_LR,
            "lr_decay_global": LR_DECAY_GLOBAL,
            "lr_decay_local":  LR_DECAY_LOCAL,
            "dropout_rate":    DROPOUT_RATE,
            "alpha":           ALPHA,
            "hidden_dim":      HIDDEN_DIM,
            "weight_decay":    WEIGHT_DECAY,
            "epochs":          EPOCHS,
            "patience":        PATIENCE,
            "region_size":     100,
            "max_patches":     MAX_SAFE_CEILING,
            "embed_dim":       EMBED_DIM,
            "n_classes":       N_CLASSES,
            "loss":            "weighted_ce + 0.5 * ordinal_mse",
            "optimizer":       "adamw + warmup_cosine + grad_clip_1.0",
            "seeds":           SEEDS,
        },
        "per_seed": [
            {
                "seed":             r["seed"],
                "qwk":              r["qwk"],
                "f1":               r["f1"],
                "confusion_matrix": r["confusion_matrix"],
            }
            for r in seed_results
        ],
        "ensemble": {
            "qwk":              ensemble_qwk,
            "f1":               ensemble_f1,
            "confusion_matrix": ensemble_cm.tolist(),
        }
    }
    with open(f"{OUTPUT_DIR}/final_ensemble_results.json", "w") as f:
        json.dump(final_record, f, indent=2)

    # Predictions file — used by heatmap/demo script
    # Contains per-patient: true label, ensemble prediction, per-seed probs, coords
    predictions_record = {
        "patient_ids":      pids_0,
        "true_labels":      true_labels,
        "ensemble_preds":   ensemble_preds,
        "ensemble_probs":   ensemble_probs.tolist(),
        "coords":           seed_results[0]["coords"],  # same for all seeds
        "grade_map":        {"0": "G1", "1": "G2", "2": "G3"},
    }
    with open(f"{OUTPUT_DIR}/val_predictions.json", "w") as f:
        json.dump(predictions_record, f, indent=2)

    print(f"\nSaved to {OUTPUT_DIR}/")
    print(f"  final_ensemble_results.json")
    print(f"  val_predictions.json")
    for seed in SEEDS:
        print(f"  seed{seed}_weights.msgpack")

    # ── Auto-download ─────────────────────────────────────────────────────
    from IPython.display import FileLink, display, HTML
    shutil.make_archive("hmil_final_models", 'zip', OUTPUT_DIR)
    display(FileLink("hmil_final_models.zip"))
    display(HTML("""
    <script>
    setTimeout(function() {
        var links = document.querySelectorAll('a[href*="hmil_final_models.zip"]');
        if (links.length > 0) links[0].click();
    }, 2000);
    </script>
    """))

Backend: tpu  |  Devices: 8
Split: 401 train / 101 val

H-MIL FINAL TRAINING — 3 SEEDS
Config: lr=2.808e-04  dr=0.183  alpha=0.5
        global_decay=0.624  local_decay=0.734
Epochs: 60  Patience: 10

#################################################################
SEED 7
#################################################################
Class weights → G1:2.59  G2:0.55  G3:1.27
Ep 01/60 | F1:0.3125 | QWK:0.0111 | Best:0.0111 | Pat:10
Ep 02/60 | F1:0.3765 | QWK:0.2283 | Best:0.2283 | Pat:10
Ep 03/60 | F1:0.5898 | QWK:0.4695 | Best:0.4695 | Pat:10
Ep 04/60 | F1:0.5591 | QWK:0.4304 | Best:0.4695 | Pat:10
Ep 05/60 | F1:0.5608 | QWK:0.4474 | Best:0.4695 | Pat:9
Ep 06/60 | F1:0.4853 | QWK:0.3610 | Best:0.4695 | Pat:8
Ep 07/60 | F1:0.5361 | QWK:0.3864 | Best:0.4695 | Pat:7
Ep 08/60 | F1:0.4888 | QWK:0.3641 | Best:0.4695 | Pat:6
Ep 09/60 | F1:0.5272 | QWK:0.3441 | Best:0.4695 | Pat:5
Ep 10/60 | F1:0.5443 | QWK:0.3725 | Best:0.4695 | Pat:4
Ep 11/60 | F1:0.5756 | QWK:0.4013 | Best:0.4695 | Pat:

/kaggle/working/hmil_final_models.zip